# ❤️ Heart Disease Predictor — Streamlit app in Google Colab

Run the cells from top to bottom:

1. Install Streamlit
2. Upload your `model.pkl`
3. Write the app file (`app.py`)
4. Start the app and get a public link

At the end you get a link like `https://something.trycloudflare.com` — click it to open your app.

### Step 1: Install Streamlit

In [ ]:
!pip install -q streamlit

### Step 2: Upload `model.pkl`

Choose the `model.pkl` you downloaded from your Random Forest notebook.

(If you trained the model in this same Colab session, the file is already here and this cell just confirms it.)

In [ ]:
import os

if os.path.exists("model.pkl"):
    print("model.pkl already here ✅")
else:
    from google.colab import files
    uploaded = files.upload()   # pick model.pkl
    print("Uploaded:", list(uploaded.keys()))

### Step 3: Check scikit-learn version

Your model was saved with **scikit-learn 1.6.1**. If Colab shows a different version and you get errors in Step 5, run
`!pip install -q scikit-learn==1.6.1`, then **Runtime → Restart session**, and run the cells again.

In [ ]:
import sklearn
print("scikit-learn version:", sklearn.__version__)

### Step 4: Write the Streamlit app

`%%writefile` saves this cell as a file called `app.py`.
(Don't name it `streamlit.py` — that would hide the real Streamlit library.)

In [ ]:
%%writefile app.py
import pickle

import pandas as pd
import streamlit as st

# ---------------------------------------------------------
# Page setup
# ---------------------------------------------------------
st.set_page_config(page_title="Heart Disease Predictor", page_icon="❤️", layout="centered")

st.title("❤️ Heart Disease Predictor")
st.write("Random Forest model trained on the UCI Heart Disease dataset (303 patients, 100 trees).")
st.caption("Student project for learning only. This is not medical advice.")

# ---------------------------------------------------------
# Load the model once (cached so it doesn't reload on every click)
# ---------------------------------------------------------
@st.cache_resource
def load_model():
    with open("model.pkl", "rb") as f:
        return pickle.load(f)

try:
    model = load_model()
except FileNotFoundError:
    st.error("model.pkl not found. Put it in the same folder as app.py.")
    st.stop()

# Same column order the model was trained on
FEATURES = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
            "thalach", "exang", "oldpeak", "slope", "ca", "thal"]

# ---------------------------------------------------------
# Input form
# ---------------------------------------------------------
st.subheader("Patient details")

with st.form("patient_form"):
    col1, col2 = st.columns(2)

    with col1:
        age = st.number_input("Age", min_value=1, max_value=120, value=52)
        sex = st.selectbox("Sex", options=[1, 0],
                           format_func=lambda x: "Male" if x == 1 else "Female")
        cp = st.selectbox("Chest pain type (cp)", options=[0, 1, 2, 3], index=2,
                          format_func=lambda x: {0: "0 - Typical angina",
                                                 1: "1 - Atypical angina",
                                                 2: "2 - Non-anginal pain",
                                                 3: "3 - Asymptomatic"}[x])
        trestbps = st.number_input("Resting blood pressure (mm Hg)",
                                   min_value=80, max_value=220, value=130)
        chol = st.number_input("Cholesterol (mg/dl)",
                               min_value=100, max_value=600, value=250)
        fbs = st.selectbox("Fasting blood sugar > 120 mg/dl?", options=[0, 1],
                           format_func=lambda x: "Yes" if x == 1 else "No")
        restecg = st.selectbox("Resting ECG result (restecg)", options=[0, 1, 2], index=1,
                               format_func=lambda x: {0: "0 - Normal",
                                                      1: "1 - ST-T abnormality",
                                                      2: "2 - LV hypertrophy"}[x])

    with col2:
        thalach = st.number_input("Max heart rate (thalach)",
                                  min_value=60, max_value=220, value=170)
        exang = st.selectbox("Exercise-induced angina?", options=[0, 1],
                             format_func=lambda x: "Yes" if x == 1 else "No")
        oldpeak = st.number_input("ST depression (oldpeak)",
                                  min_value=0.0, max_value=7.0, value=1.2, step=0.1)
        slope = st.selectbox("Slope of ST segment", options=[0, 1, 2], index=2,
                             format_func=lambda x: {0: "0 - Downsloping",
                                                    1: "1 - Flat",
                                                    2: "2 - Upsloping"}[x])
        ca = st.selectbox("Major vessels coloured (ca)", options=[0, 1, 2, 3, 4])
        thal = st.selectbox("Thal", options=[0, 1, 2, 3], index=2,
                            format_func=lambda x: {0: "0 - Unknown",
                                                   1: "1 - Fixed defect",
                                                   2: "2 - Normal",
                                                   3: "3 - Reversible defect"}[x])

    submitted = st.form_submit_button("Predict", use_container_width=True)

# ---------------------------------------------------------
# Prediction
# ---------------------------------------------------------
if submitted:
    patient = pd.DataFrame([[age, sex, cp, trestbps, chol, fbs, restecg,
                             thalach, exang, oldpeak, slope, ca, thal]],
                           columns=FEATURES)

    prediction = model.predict(patient)[0]
    prob_disease = model.predict_proba(patient)[0][1]  # share of trees voting "1"

    st.subheader("Result")
    if prediction == 1:
        st.error(f"Heart Disease likely — {prob_disease:.0%} of trees voted yes")
    else:
        st.success(f"No Heart Disease likely — {1 - prob_disease:.0%} of trees voted no")

    st.progress(float(prob_disease), text=f"Disease probability: {prob_disease:.0%}")

    with st.expander("See the input sent to the model"):
        st.dataframe(patient, hide_index=True)

# ---------------------------------------------------------
# Feature importance (what the model cares about most)
# ---------------------------------------------------------
with st.expander("Which features matter most?"):
    importance = (pd.DataFrame({"Feature": FEATURES,
                                "Importance": model.feature_importances_})
                  .sort_values("Importance", ascending=False)
                  .set_index("Feature"))
    st.bar_chart(importance)

### Step 5: Test that the model loads

In [ ]:
import pickle
import pandas as pd

with open("model.pkl", "rb") as f:
    model = pickle.load(f)

test = pd.DataFrame([[52, 1, 2, 130, 250, 0, 1, 170, 0, 1.2, 2, 0, 2]],
                    columns=["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
                             "thalach", "exang", "oldpeak", "slope", "ca", "thal"])
print("Model loaded ✅  Test prediction:", model.predict(test)[0])

### Step 6: Start the app

Colab can't open `localhost` in your browser, so we use **Cloudflare Tunnel** to make a free public link (no sign-up, no password).

Run the cell, wait ~10–20 seconds, then click the link it prints.
Keep this cell running while you use the app. Press the ■ stop button to shut it down.

In [ ]:
import subprocess, re, time, os

# Download the Cloudflare tunnel tool (only the first time)
if not os.path.exists("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

# Stop any old app, then start Streamlit in the background
subprocess.run(["pkill", "-f", "streamlit run"])
log = open("streamlit.log", "w")
app = subprocess.Popen(["streamlit", "run", "app.py",
                        "--server.port", "8501", "--server.headless", "true"],
                       stdout=log, stderr=log)
time.sleep(5)

# Start the tunnel and print the public link
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in tunnel.stdout:
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        print("✅ Your app is live. Open this link:")
        print(match.group(0))
        break

# Keep the cell running so the app stays online
try:
    tunnel.wait()
except KeyboardInterrupt:
    tunnel.terminate()
    app.terminate()
    print("App stopped.")

### If something goes wrong

Run the cell below to see Streamlit's messages.

- **`model.pkl not found`** → re-run Step 2.
- **Link shows an error page** → wait 10 more seconds and refresh.
- **Error loading the model** → see Step 3.

In [ ]:
!cat streamlit.log